In [ ]:
import pandas as pd

df = pd.read_csv("cs-training.csv")
df.head()


,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [ ]:
# Drop the unnamed index column if it exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Check for missing values
df.isnull().sum()

# Fill missing MonthlyIncome with the median value
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())

# Fill missing NumberOfDependents with the median value (usually 0)
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

# Confirm there are no more missing values
df.isnull().sum()



SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
dtype: int64

In [ ]:
# Define your target variable (what we're predicting)
target = "SeriousDlqin2yrs"

# Define your features (what we're using to predict)
features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberOfDependents"
]

X = df[features]
y = df[target]

X.head()


,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberOfDependents
0,0.766127,45,2,0.802982,9120.0,13,0,2.0
1,0.957151,40,0,0.121876,2600.0,4,0,1.0
2,0.658180,38,1,0.085113,3042.0,2,1,0.0
3,0.233810,30,0,0.036050,3300.0,5,0,0.0
4,0.907239,49,1,0.024926,63588.0,7,0,0.0


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)


(120000, 8) (30000, 8)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully")


Model trained successfully


In [ ]:
from sklearn.metrics import roc_auc_score

# Predict probabilities of default for the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate AUC-ROC score
auc_score = roc_auc_score(y_test, y_pred_proba)
print("AUC Score:", auc_score)


AUC Score: 0.6620658680822628


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_scaled = LogisticRegression(max_iter=1000)
model_scaled.fit(X_train_scaled, y_train)

y_pred_proba_scaled = model_scaled.predict_proba(X_test_scaled)[:, 1]
auc_scaled = roc_auc_score(y_test, y_pred_proba_scaled)
print("AUC Score (scaled):", auc_scaled)


AUC Score (scaled): 0.6659186546324078


In [ ]:
from sklearn.metrics import roc_auc_score

# Check which features matter most
import pandas as pd
coefficients = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print(coefficients)


                                feature  coefficient
2  NumberOfTime30-59DaysPastDueNotWorse     0.295806
7                    NumberOfDependents     0.065112
3                             DebtRatio    -0.000023
4                         MonthlyIncome    -0.000042
0  RevolvingUtilizationOfUnsecuredLines    -0.000051
5       NumberOfOpenCreditLinesAndLoans    -0.009798
1                                   age    -0.041494
6               NumberOfTimes90DaysLate    -0.263147


In [ ]:
# Combine both late payment features into one "ever late" indicator
df["EverLate"] = ((df["NumberOfTime30-59DaysPastDueNotWorse"] > 0) |
                   (df["NumberOfTimes90DaysLate"] > 0)).astype(int)

# Redefine features, replacing the two late-payment columns with the combined one
features_v2 = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfDependents",
    "EverLate"
]

X2 = df[features_v2]
y2 = df[target]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

model_v2 = LogisticRegression(max_iter=1000)
model_v2.fit(X2_train, y2_train)

y2_pred_proba = model_v2.predict_proba(X2_test)[:, 1]
auc_v2 = roc_auc_score(y2_test, y2_pred_proba)
print("AUC Score (v2):", auc_v2)


AUC Score (v2): 0.7887479585833422


In [ ]:
coefficients_v2 = pd.DataFrame({
    "feature": features_v2,
    "coefficient": model_v2.coef_[0]
}).sort_values(by="coefficient", ascending=False)

print(coefficients_v2)


                                feature  coefficient
6                              EverLate     2.142894
5                    NumberOfDependents     0.061381
2                             DebtRatio    -0.000012
3                         MonthlyIncome    -0.000025
0  RevolvingUtilizationOfUnsecuredLines    -0.000027
4       NumberOfOpenCreditLinesAndLoans    -0.008114
1                                   age    -0.025129
